# Deep Modular Network (DMoN) Pooling on PROTEINS

Graph Classification on PROTEINS (TUDataset): DMoN pooling ("Graph Clustering with Graph Neural Networks", https://arxiv.org/abs/2006.16904) learns a soft cluster assignment that maximizes Newman modularity, coarsening the graph twice between message-passing layers. This notebook ports the reference implementation to K3-Node: a sparse `GCNConv` first embeds every node, then `to_dense_batch`/`to_dense_adj` (differentiable w.r.t. node features via `ops.scatter`) densify the batch so two `K3-Node` `DMoNPooling` layers can alternate with `DenseGraphConv` layers, exactly as in the reference `Net`. The model trains with mini-batch SGD over `DataLoader` batches, combining the classification loss with DMoN's spectral + cluster regularization terms, and reports train/val/test loss and accuracy every epoch — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install git+http://github.com/anas-rz/k3-node/@main

# ==============================================================================
# K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "torch")

from math import ceil
import numpy as np
import keras
from keras import layers, ops

from k3_node import layers as k3_layers
from k3_node.datasets import TUDataset
from k3_node.loader import DataLoader

title = "Deep Modular Network (DMoN) Pooling on PROTEINS"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Dataset: shuffle once, then split into test/val/train (10%/10%/80%)
dataset = TUDataset(root="./data/PROTEINS", name="PROTEINS")
perm = np.random.permutation(len(dataset)).tolist()
dataset = dataset[perm][:100]
avg_num_nodes = int(ops.convert_to_numpy(dataset._data.x).shape[0] / len(dataset))

n = (len(dataset) + 9) // 10
test_dataset = dataset[:n]
val_dataset = dataset[n:2 * n]
train_dataset = dataset[2 * n:]
test_loader = DataLoader(test_dataset, batch_size=20)
val_loader = DataLoader(val_dataset, batch_size=20)
train_loader = DataLoader(train_dataset, batch_size=20)


# 2. Framework-agnostic dense-batch helpers. `batch`/`edge_index` are purely
# structural (no gradient needed), so plain numpy is used for those; but the
# node features `x` need gradients, so they're placed into the dense tensor
# with the differentiable `ops.scatter` instead of a numpy round-trip.
def _local_positions(batch_np):
    return np.arange(len(batch_np)) - np.searchsorted(batch_np, batch_np, side="left")


def to_dense_batch(x, batch):
    batch_np = ops.convert_to_numpy(batch).astype(np.int64)
    num_graphs = int(batch_np.max()) + 1 if batch_np.size else 1
    max_nodes = int(np.bincount(batch_np, minlength=num_graphs).max())
    local_index = _local_positions(batch_np)

    feat_dim = int(ops.shape(x)[-1])
    scatter_idx = np.stack([batch_np, local_index], axis=1)
    dense_x = ops.scatter(scatter_idx, x, shape=(num_graphs, max_nodes, feat_dim))

    mask = np.zeros((num_graphs, max_nodes), dtype=bool)
    mask[batch_np, local_index] = True
    return dense_x, ops.convert_to_tensor(mask), num_graphs, max_nodes


def to_dense_adj(edge_index, batch, num_graphs, max_nodes):
    edge_index_np = ops.convert_to_numpy(edge_index).astype(np.int64)
    batch_np = ops.convert_to_numpy(batch).astype(np.int64)
    local_index = _local_positions(batch_np)

    src, dst = edge_index_np[0], edge_index_np[1]
    edge_batch = batch_np[src]
    dense_adj = np.zeros((num_graphs, max_nodes, max_nodes), dtype=np.float32)
    dense_adj[edge_batch, local_index[src], local_index[dst]] = 1.0
    return ops.convert_to_tensor(dense_adj)


# 3. DMoN pooling architecture: GCNConv -> pool -> DenseGraphConv -> pool -> DenseGraphConv
class K3Net(keras.Model):
    def __init__(self, in_channels, out_channels, avg_num_nodes, hidden_channels=32):
        super().__init__()
        self.conv1 = k3_layers.GCNConv(in_channels, hidden_channels)
        num_nodes = ceil(0.5 * avg_num_nodes)
        self.pool1 = k3_layers.DMoNPooling([hidden_channels, hidden_channels], num_nodes)

        self.conv2 = k3_layers.DenseGraphConv(hidden_channels, hidden_channels)
        num_nodes = ceil(0.5 * num_nodes)
        self.pool2 = k3_layers.DMoNPooling([hidden_channels, hidden_channels], num_nodes)

        self.conv3 = k3_layers.DenseGraphConv(hidden_channels, hidden_channels)

        self.lin1 = layers.Dense(hidden_channels)
        self.lin2 = layers.Dense(out_channels)

    def call(self, x, edge_index, batch):
        x = ops.relu(self.conv1(x, edge_index))

        x_dense, mask, num_graphs, max_nodes = to_dense_batch(x, batch)
        adj = to_dense_adj(edge_index, batch, num_graphs, max_nodes)

        _, x_dense, adj, sp1, _, c1 = self.pool1(x_dense, adj, mask)
        x_dense = ops.relu(self.conv2(x_dense, adj))

        _, x_dense, adj, sp2, _, c2 = self.pool2(x_dense, adj)
        x_dense = self.conv3(x_dense, adj)

        out = ops.mean(x_dense, axis=1)
        out = ops.relu(self.lin1(out))
        out = self.lin2(out)
        # Matches the reference exactly: only spectral + cluster loss are
        # summed into the auxiliary loss, ortho_loss is discarded.
        return ops.log_softmax(out, axis=-1), sp1 + sp2 + c1 + c2


model = K3Net(dataset.num_features, dataset.num_classes, avg_num_nodes)

# Eager forward pass to build every sublayer's weights
sample = next(iter(train_loader))
_ = model(
    ops.convert_to_tensor(sample.x, dtype="float32"),
    ops.convert_to_tensor(sample.edge_index, dtype="int64"),
    ops.convert_to_tensor(sample.batch, dtype="int64"),
)

optimizer = keras.optimizers.Adam(learning_rate=0.001)
trainable_vars = model.trainable_variables

# Setup optimizer variables for the functional (JAX) backend
if backend == "jax":
    import jax
    optimizer.build(trainable_vars)
    opt_vars = [v.value for v in optimizer.variables]
    non_trainable_vars = [v.value for v in model.non_trainable_variables]


def nll_loss(log_probs, y, num_classes):
    y_one_hot = ops.one_hot(ops.cast(y, "int32"), num_classes)
    return -ops.mean(ops.sum(log_probs * y_one_hot, axis=-1))


def batch_tensors(data_batch):
    x = ops.convert_to_tensor(data_batch.x, dtype="float32")
    edge_index = ops.convert_to_tensor(data_batch.edge_index, dtype="int64")
    batch_vec = ops.convert_to_tensor(data_batch.batch, dtype="int64")
    y = ops.reshape(ops.convert_to_tensor(data_batch.y, dtype="int32"), (-1,))
    return x, edge_index, batch_vec, y


# 4. Multi-Backend Training Step
def train_step(data_batch):
    global opt_vars
    x, edge_index, batch_vec, y = batch_tensors(data_batch)
    num_classes = dataset.num_classes

    if backend == "torch":
        log_probs, aux_loss = model(x, edge_index, batch_vec)
        loss = nll_loss(log_probs, y, num_classes) + aux_loss
        loss.backward()
        grads = [v.value.grad for v in trainable_vars]
        optimizer.apply_gradients(zip(grads, trainable_vars))
        for v in trainable_vars:
            if v.value.grad is not None:
                v.value.grad.zero_()
        loss_value = float(ops.convert_to_numpy(loss))

    elif backend == "torch":
        import tensorflow as tf
        with tf.GradientTape() as tape:
            log_probs, aux_loss = model(x, edge_index, batch_vec)
            loss = nll_loss(log_probs, y, num_classes) + aux_loss
        grads = tape.gradient(loss, trainable_vars)
        optimizer.apply_gradients(zip(grads, trainable_vars))
        loss_value = float(ops.convert_to_numpy(loss))

    else:  # jax
        trainable_values = [v.value for v in trainable_vars]

        def loss_fn(params):
            (log_probs, aux_loss), _ = model.stateless_call(params, non_trainable_vars, x, edge_index, batch_vec)
            return nll_loss(log_probs, y, num_classes) + aux_loss

        loss_val, grads = jax.value_and_grad(loss_fn)(trainable_values)
        new_values, opt_vars = optimizer.stateless_apply(opt_vars, grads, trainable_values)
        for v, val in zip(trainable_vars, new_values):
            v.assign(val)
        loss_value = float(loss_val)

    return loss_value * int(ops.shape(y)[0])


def train(loader):
    total_loss = 0.0
    for data_batch in loader:
        total_loss += train_step(data_batch)
    return total_loss / len(loader.dataset)


def test(loader):
    total_loss, correct = 0.0, 0
    for data_batch in loader:
        x, edge_index, batch_vec, y = batch_tensors(data_batch)
        log_probs, aux_loss = model(x, edge_index, batch_vec)
        loss = nll_loss(log_probs, y, dataset.num_classes) + aux_loss
        total_loss += float(ops.convert_to_numpy(loss)) * int(ops.shape(y)[0])
        pred = ops.argmax(log_probs, axis=-1)
        correct += int(ops.convert_to_numpy(ops.sum(ops.cast(pred == ops.cast(y, pred.dtype), "int32"))))
    n = len(loader.dataset)
    return total_loss / n, correct / n


print(f"Training K3-Node DMoN model on {backend} backend...")
for epoch in range(1, 3):
    train_loss = train(train_loader)
    _, train_acc = test(train_loader)
    val_loss, val_acc = test(val_loader)
    test_loss, test_acc = test(test_loader)
    print(f"Epoch: {epoch:03d}, Train Loss: {train_loss:.3f}, "
          f"Train Acc: {train_acc:.3f}, Val Loss: {val_loss:.3f}, "
          f"Val Acc: {val_acc:.3f}, Test Loss: {test_loss:.3f}, "
          f"Test Acc: {test_acc:.3f}")

print("\n✓ K3-Node execution completed successfully!")
